# Exercise 1 — Open and query an IFC model

**Goal:** answer “What is in this file?” using `BuildingInformationModel`, its spatial tree, and element queries. Allow 20–25 minutes. Run cells in order with **Shift + Enter**. Binder already has `compas_ifc` installed.

This exercise starts with the larger Duplex model so searches return more than one object. It has four storeys. A count of zero is still a useful finding, not an error. Set `IFC_PATH` to `sample_building.ifc` to compare it with the tiny sample.

In [ ]:
from pathlib import Path
from compas_ifc.bim import BuildingInformationModel

IFC_PATH = Path("Duplex_A_20110907.ifc")  # CHANGE to sample_building.ifc to compare
if not IFC_PATH.is_file():
    raise FileNotFoundError(f"Missing {IFC_PATH}; check the JupyterLab file browser.")
model = BuildingInformationModel(str(IFC_PATH), load_geometries=False, rectify_placements=False)
print("Model:", model.name)

## 1. Follow the spatial tree

The tree answers **where does an object belong?** A typical route is project → site → building → storey → element. A parent is a relationship, not a geometric measurement.

**Predict:** how many storeys does the Duplex model contain? Run the cell, then compare.

In [ ]:
model.print_hierarchy(max_depth=4)
print("Storeys:", len(model.storeys))
for storey in model.storeys:
    print(storey.name, "—", len(list(storey.children)), "direct children")

## 2. Find objects by IFC class

`get_elements_by_type` returns a list. It also includes subclasses: `IfcWall` can find an `IfcWallStandardCase`. Try changing `TYPE_TO_EXPLORE` to `IfcWindow`.

In [ ]:
for ifc_type in ("IfcWall", "IfcWindow", "IfcDoor", "IfcSlab"):
    print(ifc_type, len(model.get_elements_by_type(ifc_type)))

TYPE_TO_EXPLORE = "IfcWall"  # CHANGE to a class with a nonzero count
matches = model.get_elements_by_type(TYPE_TO_EXPLORE)
for number, item in enumerate(matches[:5]):
    print(number, item.ifc_type, item.name, item.global_id)

## 3. Name, index, and GlobalId

The number in the preview is only a position in this Python list. An object's `global_id` is the IFC identifier. Names may repeat; do not use them as unique identifiers.

In [ ]:
if not matches:
    raise ValueError("Choose a class with a nonzero count, then rerun the previous cell.")
selected = matches[0]
same_id = model.get_element_by_global_id(selected.global_id)
same_name = model.get_elements_by_name(selected.name)
print("Class:", selected.ifc_type)
print("Name:", selected.name)
print("GlobalId:", selected.global_id)
print("Found by ID:", same_id.name if same_id else "missing")
print("Objects with this exact name:", len(same_name))
print("Parent:", selected.parent.name if selected.parent else "none")

## 4. Read the object's information

`properties` is a dictionary. A nested dictionary is often a property set. Some values may be empty or absent. Report what the model contains; do not infer a missing property from appearance.

In [ ]:
import json

print(json.dumps(selected.properties, indent=2, default=str))
wall_common = selected.properties.get("Pset_WallCommon", {})
print("IsExternal:", wall_common.get("IsExternal", "not recorded"))
print("LoadBearing:", wall_common.get("LoadBearing", "not recorded"))

## Your turn

1. Change `TYPE_TO_EXPLORE` to `IfcWindow` and rerun from that cell. Record its Name, GlobalId, and parent.
2. Which values describe identity, which describe a relationship, and which are authored properties?
3. If your larger model has many walls, show only the first five. Explain why a class search is easier than reading every entity.

**Checkpoint:** another student should be able to find the same object using the GlobalId you recorded. [API reference](https://compas.dev/compas_ifc/latest/tutorials/01_open_a_model.html).